In [16]:
import pandas as pd
import numpy as np

### Информация о пассажирах

* Пол пассажира (женский/мужской)
* Тип клиента Тип пассажира (лояльный/нелояльный)
* Возраст Возраст пассажира в годах (числовой)
* Тип путешествия
* Класс полета Бизнес/Эко/Эко Плюс
* Расстояние полета

### Рейтинги качества обслуживания

* Услуга Wi-Fi в полете
* Удобное время отправления/прибытия
* Простота онлайн-бронирования
* Расположение ворот
* Продукты питания
* Онлайн-интернат
* Комфорт сиденья
* Развлечения в полете
* Бортовое обслуживание
* Обслуживание комнаты для ног
* Обработка багажа
* Услуга регистрации
* Чистота

### Рейтинги задержки

* Удовлетворенность задержкой вылета (1–5 или 0, если не оценено)
* Удовлетворенность задержкой прибытия (1–5 или 0, если не оценено)

### Целевая переменная
* удовлетворение	Удовлетворенность пассажиров (Правда = Удовлетворенность, Ложь = Не Удовлетворенность)

In [17]:
df = pd.read_csv('avia_linear.csv', sep=',', encoding='mac_cyrillic')

columns = df.columns
print(f"Всего пропусков: {df.isnull().sum().sum()}")
print(f"Всего (строк, столбцов): {df.shape}")

Всего пропусков: 393
Всего (строк, столбцов): (129880, 22)


### Анализ пропусков

Количество пропусков (393) несоизмеримо мало с количеством данных (129.880), что составляет около 0.3% от общего количества, поэтому данные строки можно удалить и сразу преобразовать качественные значения в количественные

In [18]:
df = df.dropna(subset=['Arrival Delay in Minutes'])

df['Gender'] = np.where(df['Gender'] == 'Male', 1, 0)
df['Customer Type'] = np.where(df['Customer Type'] == 'Loyal Customer', 1, 0)
df['Type of Travel'] = np.where(df['Type of Travel'] == 'Business travel', 1, 0)
df['satisfaction'] = np.where(df['satisfaction'] == True, 1, 0)
df['Class'] = np.where(
    df['Class'] == 'Business', 2, 
    np.where(df['Class'] == 'Eco Plus', 1, 0)
)

# делим на признаки (x) и целевую переменную (y)
x = df.drop(columns=['Arrival Delay in Minutes']).to_numpy(dtype=float)
y = df['Arrival Delay in Minutes'].to_numpy(dtype=float)

print(f"Размерность х: {x.shape}")
print(f"Размерность y: {y.shape}")

Размерность х: (129487, 21)
Размерность y: (129487,)


In [19]:
def train_test_split(x, y, test_size, random_state=42):
    np.random.seed(random_state)
    indices = np.random.permutation(len(x))
    test_set_size = int(len(x) * test_size)
    
    test_indices = indices[:test_set_size]
    train_indices = indices[test_set_size:]
    
    return x[train_indices], x[test_indices], y[train_indices], y[test_indices]

x_train, x_test, y_train, y_test = train_test_split(x, y, test_size=0.2)
print(len(x_train), len(x_test))

103590 25897


In [20]:
class CustomStandardScaler:
    def fit(self, X):
        self.mean = np.mean(X, axis=0)
        self.std = np.std(X, axis=0)
        self.std[self.std == 0] = 1.0
        return self

    def transform(self, X): # маштабирование данных
        return (X - self.mean) / self.std

    def fit_transform(self, X):
        return self.fit(X).transform(X)


scaler = CustomStandardScaler()
x_train_scaler = scaler.fit_transform(x_train)
x_test_scaler = scaler.transform(x_test)

In [ ]:
class CustomLinearRegression:
    def __init__(self, fit_intercept=True):
        self.fit_intercept = fit_intercept # флаг, указывающий, нужно ли добавлять свободный член w_0 в модель
        self.weights = None # массив весов

    def fit(self, X, y):
        if self.fit_intercept:
            ones = np.ones((X.shape[0], 1))
            X = np.hstack([ones, X]) # приклеивание вектора из единиц для свободного члена
        
        # Формула аналитического решения (OLS): w = (X^T * X)^(-1) * X^T * y
        self.weights = np.linalg.inv(X.T @ X) @ X.T @ y
        return self

    def predict(self, X):
        if self.fit_intercept:
            ones = np.ones((X.shape[0], 1))
            X = np.hstack([ones, X])
            
        return X @ self.weights

In [22]:
def smse(y_true, y_pred):
    return np.sqrt(np.mean((y_true - y_pred) ** 2))

def mae(y_true, y_pred):
    return np.mean(np.abs(y_true - y_pred))

def r2(y_true, y_pred):
    ss_res = np.sum((y_true - y_pred) ** 2)
    ss_tot = np.sum((y_true - np.mean(y_true)) ** 2)
    return 1 - (ss_res / ss_tot)

In [25]:
model = CustomLinearRegression(fit_intercept=True)
model.fit(x_train_scaler, y_train)

y_train_pred = model.predict(x_train_scaler)
y_test_pred = model.predict(x_test_scaler)


print("Метрики на обучающей выборке (Train)")
print(f"RMSE: {smse(y_train, y_train_pred):.4f}")
print(f"MAE:  {mae(y_train, y_train_pred):.4f}")
print(f"R2:   {r2(y_train, y_train_pred):.4f}")

print()
print("Метрики на тестовой выборке (Test)")
print(f"RMSE: {smse(y_test, y_test_pred):.4f}")
print(f"MAE:  {mae(y_test, y_test_pred):.4f}")
print(f"R2:   {r2(y_test, y_test_pred):.4f}")

Метрики на обучающей выборке (Train)
RMSE: 10.0504
MAE:  5.2603
R2:   0.9289

Метрики на тестовой выборке (Test)
RMSE: 9.9777
MAE:  5.3320
R2:   0.9419
